# Chapter 4 — Matrix Decompositions
## Mathematics for Machine Learning (Deisenroth, Faisal & Ong)
### Complete Exercise Solutions (4.1 – 4.12)

> **Note on Source Material:**  
> All exercises and theoretical formulations in this notebook are taken directly from the textbook  
> **"Mathematics for Machine Learning"** by Marc Peter Deisenroth, A. Aldo Faisal, and Cheng Soon Ong (Cambridge University Press).


---


In [1]:
import numpy as np
from numpy.linalg import eig, svd, det, inv, matrix_rank
import sympy as sp
from sympy import Matrix, symbols, Rational, sqrt, simplify, eye, pprint, oo
from sympy import init_printing
init_printing(use_unicode=True)
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

---
## Exercise 4.1 — Determinant via Laplace Expansion & Sarrus Rule

In [2]:
print("=== Exercise 4.1 ===")
A = sp.Matrix([[1, 3, 5], [2, 4, 6], [0, 2, 4]])
print("A ="); pprint(A)

# Laplace expansion along first row
print("\nLaplace expansion (first row):")
C11 = sp.Matrix([[4, 6], [2, 4]]).det()
C12 = -sp.Matrix([[2, 6], [0, 4]]).det()
C13 = sp.Matrix([[2, 4], [0, 2]]).det()
det_laplace = 1*C11 + 3*C12 + 5*C13
print(f"  1·({C11}) + 3·({C12}) + 5·({C13}) = {det_laplace}")

# Sarrus rule
print("\nSarrus rule:")
a = A.tolist()
pos = a[0][0]*a[1][1]*a[2][2] + a[0][1]*a[1][2]*a[2][0] + a[0][2]*a[1][0]*a[2][1]
neg = a[0][2]*a[1][1]*a[2][0] + a[0][1]*a[1][0]*a[2][2] + a[0][0]*a[1][2]*a[2][1]
det_sarrus = pos - neg
print(f"  Positive diagonals: {pos}")
print(f"  Negative diagonals: {neg}")
print(f"  det = {pos} - {neg} = {det_sarrus}")

print(f"\nSymPy det: {A.det()}")
print(f"All methods agree: {det_laplace == det_sarrus == A.det()} ✓")

=== Exercise 4.1 ===
A =
⎡1  3  5⎤
⎢       ⎥
⎢2  4  6⎥
⎢       ⎥
⎣0  2  4⎦

Laplace expansion (first row):
  1·(4) + 3·(-8) + 5·(4) = 0

Sarrus rule:
  Positive diagonals: 36
  Negative diagonals: 36
  det = 36 - 36 = 0

SymPy det: 0
All methods agree: True ✓


---
## Exercise 4.2 — Efficient 5×5 Determinant

In [3]:
print("=== Exercise 4.2 ===")
A42 = sp.Matrix([
    [2, 0, 1, 2, 0],
    [2, -1, 0, 1, 1],
    [0, 1, 2, 1, 2],
    [-2, 0, 2, -1, 2],
    [2, 0, 0, 1, 1]
])
print("A ="); pprint(A42)

# Row reduce to upper triangular (tracking sign changes)
print("\nUsing row operations to triangularize:")
print(f"det(A) = {A42.det()}")

# Show the step-by-step via LU-like decomposition
L, U, perm = A42.LUdecomposition()
print("\nU (upper triangular from LU):")
pprint(U)
print(f"det(U) = product of diagonal = {sp.prod([U[i,i] for i in range(5)])}")
print(f"det(L) = 1 (unit lower triangular)")
print(f"Number of row swaps: {len(perm)}")
sign = (-1)**len(perm)
print(f"det(A) = ({sign}) × det(L) × det(U) = {sign * sp.prod([U[i,i] for i in range(5)])}")

=== Exercise 4.2 ===
A =
⎡2   0   1  2   0⎤
⎢                ⎥
⎢2   -1  0  1   1⎥
⎢                ⎥
⎢0   1   2  1   2⎥
⎢                ⎥
⎢-2  0   2  -1  2⎥
⎢                ⎥
⎣2   0   0  1   1⎦

Using row operations to triangularize:
det(A) = 6

U (upper triangular from LU):
⎡2  0   1   2   0 ⎤
⎢                 ⎥
⎢0  -1  -1  -1  1 ⎥
⎢                 ⎥
⎢0  0   1   0   3 ⎥
⎢                 ⎥
⎢0  0   0   1   -7⎥
⎢                 ⎥
⎣0  0   0   0   -3⎦
det(U) = product of diagonal = 6
det(L) = 1 (unit lower triangular)
Number of row swaps: 0
det(A) = (1) × det(L) × det(U) = 6


---
## Exercise 4.3 — Eigenspaces of 2×2 Matrices

In [4]:
print("=== Exercise 4.3 ===")

matrices = [
    sp.Matrix([[-2, 2], [1, 1]]),
    sp.Matrix([[1, 0], [2, 1]])
]

for i, M in enumerate(matrices):
    print(f"\n--- Matrix {i+1} ---")
    pprint(M)
    eigendata = M.eigenvects()
    for eigenval, mult, eigvecs in eigendata:
        print(f"  λ = {eigenval} (multiplicity {mult})")
        print(f"  Eigenspace: span{[v.T for v in eigvecs]}")

=== Exercise 4.3 ===

--- Matrix 1 ---
⎡-2  2⎤
⎢     ⎥
⎣1   1⎦
  λ = -1/2 + sqrt(17)/2 (multiplicity 1)
  Eigenspace: span[Matrix([[-3/2 + sqrt(17)/2, 1]])]
  λ = -sqrt(17)/2 - 1/2 (multiplicity 1)
  Eigenspace: span[Matrix([[-sqrt(17)/2 - 3/2, 1]])]

--- Matrix 2 ---
⎡1  0⎤
⎢    ⎥
⎣2  1⎦
  λ = 1 (multiplicity 2)
  Eigenspace: span[Matrix([[0, 1]])]


---
## Exercise 4.4 — Eigenspaces of a 4×4 Matrix

In [5]:
print("=== Exercise 4.4 ===")
A44 = sp.Matrix([
    [0, -1, 1, 1],
    [-1, 1, -2, 3],
    [2, -1, 0, 0],
    [1, -1, 1, 0]
])
print("A ="); pprint(A44)
print(f"\nCharacteristic polynomial: {A44.charpoly().as_expr()}")

eigendata = A44.eigenvects()
for eigenval, mult, eigvecs in eigendata:
    print(f"\nλ = {eigenval} (algebraic multiplicity = {mult}, geometric multiplicity = {len(eigvecs)})")
    for j, v in enumerate(eigvecs):
        print(f"  Eigenvector {j+1}: {v.T}")
    print(f"  Eigenspace E_{eigenval} = span{[v.T for v in eigvecs]}")

=== Exercise 4.4 ===
A =
⎡0   -1  1   1⎤
⎢             ⎥
⎢-1  1   -2  3⎥
⎢             ⎥
⎢2   -1  0   0⎥
⎢             ⎥
⎣1   -1  1   0⎦

Characteristic polynomial: lambda**4 - lambda**3 - 3*lambda**2 + lambda + 2

λ = -1 (algebraic multiplicity = 2, geometric multiplicity = 1)
  Eigenvector 1: Matrix([[0, 1, 1, 0]])
  Eigenspace E_-1 = span[Matrix([[0, 1, 1, 0]])]

λ = 1 (algebraic multiplicity = 1, geometric multiplicity = 1)
  Eigenvector 1: Matrix([[1, 1, 1, 1]])
  Eigenspace E_1 = span[Matrix([[1, 1, 1, 1]])]

λ = 2 (algebraic multiplicity = 1, geometric multiplicity = 1)
  Eigenvector 1: Matrix([[1, 0, 1, 1]])
  Eigenspace E_2 = span[Matrix([[1, 0, 1, 1]])]


---
## Exercise 4.5 — Diagonalizability vs. Invertibility

In [6]:
print("=== Exercise 4.5 ===")
matrices_45 = [
    ("I", sp.Matrix([[1, 0], [0, 1]])),
    ("diag(1,0)", sp.Matrix([[1, 0], [0, 0]])),
    ("upper tri", sp.Matrix([[1, 1], [0, 1]])),
    ("nilpotent", sp.Matrix([[0, 1], [0, 0]]))
]

print(f"{'Matrix':<12} | {'Diagonalizable?':<16} | {'Invertible?':<12} | Eigenvalues")
print("-" * 70)
for name, M in matrices_45:
    is_diag = M.is_diagonalizable()
    is_inv = M.det() != 0
    evals = list(M.eigenvals().keys())
    print(f"{name:<12} | {str(is_diag):<16} | {str(is_inv):<12} | {evals}")

print("\n→ Diagonalizability and invertibility are INDEPENDENT properties.")

=== Exercise 4.5 ===
Matrix       | Diagonalizable?  | Invertible?  | Eigenvalues
----------------------------------------------------------------------
I            | True             | True         | [1]
diag(1,0)    | True             | False        | [1, 0]
upper tri    | False            | True         | [1]
nilpotent    | False            | False        | [0]

→ Diagonalizability and invertibility are INDEPENDENT properties.


---
## Exercise 4.6 — Eigenspaces and Diagonalizability

In [7]:
print("=== Exercise 4.6 ===")

# (a)
print("--- (a) ---")
A46a = sp.Matrix([[2, 3, 0], [1, 4, 3], [0, 0, 1]])
pprint(A46a)
print(f"Eigenvalues: {A46a.eigenvals()}")
for eigenval, mult, eigvecs in A46a.eigenvects():
    print(f"  λ={eigenval}: alg.mult={mult}, geom.mult={len(eigvecs)}")
print(f"Diagonalizable? {A46a.is_diagonalizable()}")

# (b)
print("\n--- (b) ---")
A46b = sp.Matrix([[1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
pprint(A46b)
print(f"Eigenvalues: {A46b.eigenvals()}")
for eigenval, mult, eigvecs in A46b.eigenvects():
    print(f"  λ={eigenval}: alg.mult={mult}, geom.mult={len(eigvecs)}")
print(f"Diagonalizable? {A46b.is_diagonalizable()}")

=== Exercise 4.6 ===
--- (a) ---
⎡2  3  0⎤
⎢       ⎥
⎢1  4  3⎥
⎢       ⎥
⎣0  0  1⎦
Eigenvalues: {1: 2, 5: 1}
  λ=1: alg.mult=2, geom.mult=1
  λ=5: alg.mult=1, geom.mult=1
Diagonalizable? False

--- (b) ---
⎡1  1  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎣0  0  0  0⎦
Eigenvalues: {0: 3, 1: 1}
  λ=0: alg.mult=3, geom.mult=3
  λ=1: alg.mult=1, geom.mult=1
Diagonalizable? True


---
## Exercise 4.7 — Diagonalization (Multiple Matrices)

In [8]:
print("=== Exercise 4.7 ===")

matrices_47 = [
    ("(a)", sp.Matrix([[1, 4], [-2, -5]])),
    ("(b)", sp.Matrix([[2, 3, 0], [1, 4, 3], [0, 0, 1]])),  # same as 4.6a
    ("(c)", sp.Matrix([[5, 4, 2, 1], [0, 1, -1, -1], [-1, -1, 3, 0], [1, 1, -1, 2]])),
    ("(d)", sp.Matrix([[5, -6, -6], [-1, 4, 2], [3, -6, -4]]))
]

for name, M in matrices_47:
    print(f"\n{'='*40}")
    print(f"--- {name} ---")
    pprint(M)
    print(f"Eigenvalues: {M.eigenvals()}")
    is_diag = M.is_diagonalizable()
    print(f"Diagonalizable? {is_diag}")
    
    if is_diag:
        P, D = M.diagonalize()
        print("D (diagonal form):")
        pprint(D)
        print("P (basis matrix):")
        pprint(P)
        print(f"Verify P⁻¹AP = D: {simplify(P.inv() * M * P) == D}")
    else:
        print("Cannot diagonalize.")
        for eigenval, mult, eigvecs in M.eigenvects():
            print(f"  λ={eigenval}: alg.mult={mult}, geom.mult={len(eigvecs)} → ", end="")
            print("DEFICIENT" if len(eigvecs) < mult else "OK")

=== Exercise 4.7 ===

--- (a) ---
⎡1   4 ⎤
⎢      ⎥
⎣-2  -5⎦
Eigenvalues: {-1: 1, -3: 1}
Diagonalizable? True
D (diagonal form):
⎡-3  0 ⎤
⎢      ⎥
⎣0   -1⎦
P (basis matrix):
⎡-1  -2⎤
⎢      ⎥
⎣1   1 ⎦
Verify P⁻¹AP = D: True

--- (b) ---
⎡2  3  0⎤
⎢       ⎥
⎢1  4  3⎥
⎢       ⎥
⎣0  0  1⎦
Eigenvalues: {1: 2, 5: 1}
Diagonalizable? False
Cannot diagonalize.
  λ=1: alg.mult=2, geom.mult=1 → DEFICIENT
  λ=5: alg.mult=1, geom.mult=1 → OK

--- (c) ---
⎡5   4   2   1 ⎤
⎢              ⎥
⎢0   1   -1  -1⎥
⎢              ⎥
⎢-1  -1  3   0 ⎥
⎢              ⎥
⎣1   1   -1  2 ⎦
Eigenvalues: {2: 1, 1: 1, 4: 2}
Diagonalizable? False
Cannot diagonalize.
  λ=1: alg.mult=1, geom.mult=1 → OK
  λ=2: alg.mult=1, geom.mult=1 → OK
  λ=4: alg.mult=2, geom.mult=1 → DEFICIENT

--- (d) ---
⎡5   -6  -6⎤
⎢          ⎥
⎢-1  4   2 ⎥
⎢          ⎥
⎣3   -6  -4⎦
Eigenvalues: {1: 1, 2: 2}
Diagonalizable? True
D (diagonal form):
⎡1  0  0⎤
⎢       ⎥
⎢0  2  0⎥
⎢       ⎥
⎣0  0  2⎦
P (basis matrix):
⎡3   2  2⎤
⎢        ⎥
⎢-1  1  0⎥


---
## Exercise 4.8 — SVD of a 2×2 Matrix

In [9]:
print("=== Exercise 4.8 ===")
A48 = sp.Matrix([[3, 2], [2, 2]])
print("A ="); pprint(A48)

# A^T A
AtA = A48.T * A48
print("\nA^T A ="); pprint(AtA)

# Eigenvalues of A^T A = singular values squared
eig_AtA = AtA.eigenvects()
print("\nEigenvalues of A^T A (= σ²):")
singular_vals = []
V_cols = []
for eigenval, mult, eigvecs in sorted(eig_AtA, key=lambda x: -x[0]):
    sigma = sp.sqrt(eigenval)
    print(f"  σ² = {eigenval}, σ = {sigma}")
    singular_vals.append(sigma)
    V_cols.append(eigvecs[0].normalized())

# V matrix
V = V_cols[0].row_join(V_cols[1])
print("\nV ="); pprint(V)

# Sigma
Sigma = sp.diag(*singular_vals)
print("\nΣ ="); pprint(Sigma)

# U = A V Σ^{-1}
U_cols = []
for i in range(len(singular_vals)):
    u_i = (A48 * V_cols[i]) / singular_vals[i]
    U_cols.append(simplify(u_i))
U = U_cols[0].row_join(U_cols[1])
print("\nU ="); pprint(simplify(U))

# Verify
reconstructed = simplify(U * Sigma * V.T)
print("\nU Σ V^T ="); pprint(reconstructed)
print(f"Matches A? {reconstructed == A48}")

# NumPy verification
print("\nNumPy SVD verification:")
U_np, S_np, Vt_np = np.linalg.svd(np.array(A48, dtype=float))
print(f"Singular values: {S_np}")

=== Exercise 4.8 ===
A =
⎡3  2⎤
⎢    ⎥
⎣2  2⎦

A^T A =
⎡13  10⎤
⎢      ⎥
⎣10  8 ⎦

Eigenvalues of A^T A (= σ²):
  σ² = 5*sqrt(17)/2 + 21/2, σ = sqrt(5*sqrt(17)/2 + 21/2)
  σ² = 21/2 - 5*sqrt(17)/2, σ = sqrt(21/2 - 5*sqrt(17)/2)

V =
⎡       1   √17                 1   √17        ⎤
⎢       ─ + ───                 ─ - ───        ⎥
⎢       4    4                  4    4         ⎥
⎢─────────────────────  ───────────────────────⎥
⎢     ________________       __________________⎥
⎢    ╱              2       ╱            2     ⎥
⎢   ╱      ⎛1   √17⎞       ╱  ⎛  1   √17⎞      ⎥
⎢  ╱   1 + ⎜─ + ───⎟      ╱   ⎜- ─ + ───⎟  + 1 ⎥
⎢╲╱        ⎝4    4 ⎠    ╲╱    ⎝  4    4 ⎠      ⎥
⎢                                              ⎥
⎢          1                       1           ⎥
⎢─────────────────────  ───────────────────────⎥
⎢     ________________       __________________⎥
⎢    ╱              2       ╱            2     ⎥
⎢   ╱      ⎛1   √17⎞       ╱  ⎛  1   √17⎞      ⎥
⎢  ╱   1 + ⎜─ + ───⎟      ╱   ⎜-

---
## Exercise 4.9 — SVD of Another Matrix

In [10]:
print("=== Exercise 4.9 ===")
A49 = sp.Matrix([[2, 2], [-1, 1]])
print("A ="); pprint(A49)

AtA49 = A49.T * A49
AAt49 = A49 * A49.T
print("\nA^T A ="); pprint(AtA49)
print("A A^T ="); pprint(AAt49)

# Full SVD using sympy
# Eigenvalues of A^T A
print("\nEigenvalues of A^T A:")
for val, mult, vecs in sorted(AtA49.eigenvects(), key=lambda x: -x[0]):
    print(f"  σ² = {val}, σ = {sp.sqrt(val)}, v = {[simplify(v.normalized()).T for v in vecs]}")

# NumPy verification
U_np, S_np, Vt_np = np.linalg.svd(np.array(A49, dtype=float))
print(f"\nNumPy singular values: {S_np}")
print(f"U =\n{U_np}")
print(f"V^T =\n{Vt_np}")
print(f"Reconstruction = U @ diag(S) @ V^T =\n{U_np @ np.diag(S_np) @ Vt_np}")

=== Exercise 4.9 ===
A =
⎡2   2⎤
⎢     ⎥
⎣-1  1⎦

A^T A =
⎡5  3⎤
⎢    ⎥
⎣3  5⎦
A A^T =
⎡8  0⎤
⎢    ⎥
⎣0  2⎦

Eigenvalues of A^T A:
  σ² = 8, σ = 2*sqrt(2), v = [Matrix([[sqrt(2)/2, sqrt(2)/2]])]
  σ² = 2, σ = sqrt(2), v = [Matrix([[-sqrt(2)/2, sqrt(2)/2]])]

NumPy singular values: [2.8284 1.4142]
U =
[[-1.  0.]
 [ 0.  1.]]
V^T =
[[-0.7071 -0.7071]
 [-0.7071  0.7071]]
Reconstruction = U @ diag(S) @ V^T =
[[ 2.  2.]
 [-1.  1.]]


---
## Exercise 4.10 — Best Rank-1 Approximation

In [11]:
print("=== Exercise 4.10 ===")
A410 = np.array([[3, 2], [2, -2]])
print(f"A = \n{A410}")

U, S, Vt = np.linalg.svd(A410)
print(f"\nSingular values: {S}")
print(f"U = \n{U}")
print(f"V^T = \n{Vt}")

# Best rank-1 approximation: σ₁ u₁ v₁ᵀ
A_rank1 = S[0] * np.outer(U[:, 0], Vt[0, :])
print(f"\nBest rank-1 approximation:")
print(f"σ₁·u₁·v₁ᵀ = {S[0]:.4f} × {U[:,0]} ⊗ {Vt[0,:]}")
print(f"= \n{A_rank1}")

print(f"\nApproximation error (Frobenius): ||A - A₁||_F = σ₂ = {S[1]:.4f}")

=== Exercise 4.10 ===
A = 
[[ 3  2]
 [ 2 -2]]

Singular values: [3.7016 2.7016]
U = 
[[-0.9436 -0.331 ]
 [-0.331   0.9436]]
V^T = 
[[-0.9436 -0.331 ]
 [ 0.331  -0.9436]]

Best rank-1 approximation:
σ₁·u₁·v₁ᵀ = 3.7016 × [-0.9436 -0.331 ] ⊗ [-0.9436 -0.331 ]
= 
[[3.296  1.1562]
 [1.1562 0.4056]]

Approximation error (Frobenius): ||A - A₁||_F = σ₂ = 2.7016


---
## Exercise 4.11 — Same Nonzero Eigenvalues for $A^\top A$ and $AA^\top$

### Solution 4.11

**Proof:** Let $\lambda \neq 0$ be an eigenvalue of $A^\top A$ with eigenvector $v$:
$$A^\top A v = \lambda v$$

Multiply both sides by $A$:
$$A(A^\top A v) = A(\lambda v)$$
$$AA^\top (Av) = \lambda (Av)$$

Since $\lambda \neq 0$, $v \neq 0$ implies $Av \neq 0$ (otherwise $A^\top A v = A^\top 0 = 0 \neq \lambda v$).

So $Av$ is an eigenvector of $AA^\top$ with eigenvalue $\lambda$. ∎

By symmetry (replace $A$ with $A^\top$), the same holds in the other direction.

In [12]:
print("=== Exercise 4.11 — Verification ===")
A_test = np.random.randn(3, 5)
eig_AtA = np.sort(np.linalg.eigvalsh(A_test.T @ A_test))[::-1]
eig_AAt = np.sort(np.linalg.eigvalsh(A_test @ A_test.T))[::-1]
print(f"A is {A_test.shape}")
print(f"Eigenvalues of A^T A (5): {eig_AtA}")
print(f"Eigenvalues of A A^T (3): {eig_AAt}")
print(f"Nonzero eigenvalues match: {np.allclose(eig_AtA[:3], eig_AAt)} ✓")

=== Exercise 4.11 — Verification ===
A is (3, 5)
Eigenvalues of A^T A (5): [ 9.688   6.6986  3.7912  0.     -0.    ]
Eigenvalues of A A^T (3): [9.688  6.6986 3.7912]
Nonzero eigenvalues match: True ✓


---
## Exercise 4.12 — Largest Singular Value as Operator Norm

### Solution 4.12

**Show:** $\max_{x \neq 0} \frac{\|Ax\|_2}{\|x\|_2} = \sigma_1$

**Proof:** Let $A = U\Sigma V^\top$ be the SVD. Let $y = V^\top x$, so $\|y\| = \|x\|$ (orthogonal transformation preserves norms).

$$\frac{\|Ax\|^2}{\|x\|^2} = \frac{\|U\Sigma V^\top x\|^2}{\|x\|^2} = \frac{\|\Sigma y\|^2}{\|y\|^2} = \frac{\sum_i \sigma_i^2 y_i^2}{\sum_i y_i^2} \leq \sigma_1^2 \cdot \frac{\sum_i y_i^2}{\sum_i y_i^2} = \sigma_1^2$$

Equality holds when $y = e_1$ (i.e., $x = v_1$, the first right singular vector). ∎

In [13]:
print("=== Exercise 4.12 — Verification ===")
A_test = np.random.randn(4, 3)
U, S, Vt = np.linalg.svd(A_test)

# Try many random vectors
max_ratio = 0
for _ in range(100000):
    x = np.random.randn(3)
    ratio = np.linalg.norm(A_test @ x) / np.linalg.norm(x)
    max_ratio = max(max_ratio, ratio)

# Maximum at v1
v1 = Vt[0, :]
ratio_v1 = np.linalg.norm(A_test @ v1) / np.linalg.norm(v1)

print(f"σ₁ = {S[0]:.6f}")
print(f"max ||Ax||/||x|| (random search) ≈ {max_ratio:.6f}")
print(f"||Av₁||/||v₁|| = {ratio_v1:.6f}")
print(f"All equal: {np.allclose(S[0], ratio_v1)} ✓")

=== Exercise 4.12 — Verification ===
σ₁ = 4.069486
max ||Ax||/||x|| (random search) ≈ 4.069455
||Av₁||/||v₁|| = 4.069486
All equal: True ✓
